# An unavailable input stays visible

This uses an existing owned AAPL execution and deliberately requests an unavailable table from its metadata endpoint. Nothing is authored, executed or substituted. For interactive question/reconnect controls, see the existing [clarification walkthrough](../demos/08_clarification_and_reconnect.ipynb).

In [ ]:
from report_helpers import connect, task
from stratifyx.core.api_error import ApiError

epoch = connect()
review = task('aapl_order_activity', attempt='demo-01')

## 1. Inspect supported inputs

An empty selection only discovers the actual table names. It does not bind every table or execute work.

In [ ]:
catalogue = await epoch.extension_inputs(review.source_execution_id, tables=[])
print('Available tables:', [table.table for table in catalogue.tables])

## 2. Observe an explicit refusal

The following name is intentionally invalid for this case. A permission, login or network error is not accepted as the expected unavailable-table outcome.

In [ ]:
try:
    await epoch.extension_inputs(review.source_execution_id, tables=['__not_a_source_table__'])
except ApiError as error:
    assert error.status_code == 400, 'Inspect the actual service error; do not treat it as missing data.'
    print('Expected unavailable-table refusal: HTTP', error.status_code)
else:
    raise AssertionError('The deliberately unavailable table was unexpectedly accepted.')

## 3. Close this client

No alternative source or made-up result was used.

In [ ]:
await epoch.aclose()